# Bank Loan Eligibility Prediction Using Statistical Analysis and Machine Learning
**SI Lab project — Google Colab edition**

This notebook studies applicant data, performs statistical practicals, trains classification models, evaluates them, and provides an interactive form to estimate whether an applicant is **Likely Eligible** or **Not Likely Eligible**.

> **Important:** The default data is synthetic for demonstration. For meaningful results, upload a real loan approval dataset with a target such as `Loan_Status` (`Y` = approved, `N` = not approved). This is an academic model, not an actual bank decision.

## Practical 1 — Import libraries and load data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats
import ipywidgets as widgets

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully.")

### Load your dataset (optional)

When prompted, upload your CSV file. For the common Kaggle Loan Prediction dataset, the target column is usually `Loan_Status`. If you skip upload, the notebook creates a synthetic dataset so every practical can run.

In [ ]:
uploaded = files.upload()  # Select your CSV file; press Cancel/skip if you do not have one.
if uploaded:
    filename = next(iter(uploaded))
    raw_df = pd.read_csv(filename)
    print("Loaded:", filename, "| Shape:", raw_df.shape)
else:
    raw_df = None

In [ ]:
def make_demo_data(n=1800, seed=42):
    rng = np.random.default_rng(seed)
    gender = rng.choice(["Male", "Female"], n, p=[0.68, 0.32])
    married = rng.choice(["Yes", "No"], n, p=[0.62, 0.38])
    dependents = rng.choice(["0", "1", "2", "3+"], n, p=[0.55, 0.18, 0.17, 0.10])
    education = rng.choice(["Graduate", "Not Graduate"], n, p=[0.78, 0.22])
    self_employed = rng.choice(["No", "Yes"], n, p=[0.86, 0.14])
    applicant_income = np.clip(rng.lognormal(np.log(5200), .55, n), 800, 30000).round()
    coapplicant_income = np.where(rng.random(n) < .35, 0, np.clip(rng.lognormal(np.log(1800), .7, n), 100, 15000)).round()
    loan_amount = np.clip(rng.normal(145, 65, n), 20, 500).round()
    loan_term = rng.choice([120, 180, 240, 300, 360, 480], n, p=[.03,.06,.08,.12,.62,.09])
    credit_history = rng.choice([0, 1], n, p=[.18, .82])
    property_area = rng.choice(["Rural", "Semiurban", "Urban"], n, p=[.28, .38, .34])
    total_income = applicant_income + coapplicant_income
    logit = (-1.15 + 2.1*credit_history + .00008*(total_income-6000)
             - .004*(loan_amount-140) + .28*(education=="Graduate")
             + .30*(property_area=="Semiurban") - .25*(self_employed=="Yes"))
    p = 1/(1+np.exp(-logit))
    status = np.where(rng.binomial(1, p) == 1, "Y", "N")
    df = pd.DataFrame({
        "Loan_ID": [f"LP{100000+i}" for i in range(n)],
        "Gender": gender, "Married": married, "Dependents": dependents,
        "Education": education, "Self_Employed": self_employed,
        "ApplicantIncome": applicant_income, "CoapplicantIncome": coapplicant_income,
        "LoanAmount": loan_amount, "Loan_Amount_Term": loan_term,
        "Credit_History": credit_history, "Property_Area": property_area,
        "Loan_Status": status
    })
    # Add a few missing values to demonstrate cleaning.
    for col in ["Gender", "Married", "LoanAmount", "Credit_History"]:
        idx = rng.choice(df.index, size=max(1, int(.025*n)), replace=False)
        df.loc[idx, col] = np.nan
    return df

if raw_df is None:
    raw_df = make_demo_data()
    print("Using synthetic demonstration data.")
df = raw_df.copy()
print("Dataset shape:", df.shape)
display(df.head())

## Practical 2 — Data cleaning and preprocessing

In [ ]:
# Standardize column names but preserve the original dataframe in raw_df.
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]
print("Columns:", df.columns.tolist())
print("\nMissing values before cleaning:")
display(df.isna().sum().to_frame("missing_count").query("missing_count > 0"))

# Find a supported loan-approval target column.
target_candidates = ["Loan_Status", "loan_status", "loan_approval", "LoanApproval", "Approved", "approved", "target"]
TARGET = next((c for c in target_candidates if c in df.columns), None)
if TARGET is None:
    raise ValueError("Could not find loan approval target. Expected a column such as Loan_Status with Y/N values.")

def map_approval(value):
    if pd.isna(value):
        return np.nan
    v = str(value).strip().lower()
    if v in {"y", "yes", "approved", "approve", "1", "true", "eligible"}:
        return 1
    if v in {"n", "no", "rejected", "reject", "0", "false", "not eligible", "ineligible"}:
        return 0
    try:
        num = float(v)
        if num in (0, 1):
            return int(num)
    except Exception:
        pass
    return np.nan

df["Eligibility"] = df[TARGET].apply(map_approval)
print("Target mapping: 1 = approved/eligible; 0 = not approved/not eligible")
display(df["Eligibility"].value_counts(dropna=False).to_frame("count"))

# Drop rows where target cannot be interpreted; impute feature missing values later inside model pipelines.
df = df.dropna(subset=["Eligibility"]).copy()
df["Eligibility"] = df["Eligibility"].astype(int)
print("Shape after target cleaning:", df.shape)
print("\nMissing values remaining in features:")
display(df.isna().sum().to_frame("missing_count").query("missing_count > 0"))

## Practical 3 — Descriptive statistics

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if not numeric_df.empty:
    summary = numeric_df.describe().T
    summary["median"] = numeric_df.median(numeric_only=True)
    summary["mode"] = numeric_df.mode().iloc[0]
    summary["variance"] = numeric_df.var(numeric_only=True)
    summary["standard_deviation"] = numeric_df.std(numeric_only=True)
    summary["Q1"] = numeric_df.quantile(.25)
    summary["Q3"] = numeric_df.quantile(.75)
    summary["IQR"] = summary["Q3"] - summary["Q1"]
    display(summary.round(3))
else:
    print("No numeric features found.")

## Practical 4 — Data visualization (histogram, box, bar, pie, scatter, line)

In [ ]:
# 1. Histogram
hist_col = next((c for c in ["ApplicantIncome", "Applicant_Income"] if c in df.columns), numeric_df.columns[0] if len(numeric_df.columns) else None)
if hist_col:
    plt.figure(figsize=(7,4)); plt.hist(pd.to_numeric(df[hist_col], errors="coerce").dropna(), bins=25)
    plt.title(f"Histogram of {hist_col}"); plt.xlabel(hist_col); plt.ylabel("Frequency"); plt.show()

# 2. Box plot
if hist_col:
    plt.figure(figsize=(7,3)); plt.boxplot(pd.to_numeric(df[hist_col], errors="coerce").dropna(), vert=False)
    plt.title(f"Box plot of {hist_col}"); plt.xlabel(hist_col); plt.show()

# 3. Bar chart: approval counts
df["Eligibility_Label"] = df["Eligibility"].map({1:"Eligible/Approved", 0:"Not Eligible/Rejected"})
df["Eligibility_Label"].value_counts().plot(kind="bar", figsize=(6,4), title="Loan approval class counts")
plt.ylabel("Number of applicants"); plt.xticks(rotation=0); plt.show()

# 4. Pie chart
df["Eligibility_Label"].value_counts().plot(kind="pie", autopct="%1.1f%%", figsize=(5,5), ylabel="", title="Approval share")
plt.show()

# 5. Scatter plot
income_col = next((c for c in ["ApplicantIncome", "Applicant_Income"] if c in df.columns), None)
loan_col = next((c for c in ["LoanAmount", "Loan_Amount"] if c in df.columns), None)
if income_col and loan_col:
    plt.figure(figsize=(7,4))
    plt.scatter(pd.to_numeric(df[income_col], errors="coerce"), pd.to_numeric(df[loan_col], errors="coerce"), alpha=.5)
    plt.title("Applicant income vs loan amount"); plt.xlabel(income_col); plt.ylabel(loan_col); plt.show()

# 6. Line chart: approval rate by income group (sorted bins)
if income_col:
    tmp = df[[income_col, "Eligibility"]].copy()
    tmp[income_col] = pd.to_numeric(tmp[income_col], errors="coerce")
    tmp = tmp.dropna()
    if tmp[income_col].nunique() > 1:
        tmp["Income_Group"] = pd.qcut(tmp[income_col], q=min(8, tmp[income_col].nunique()), duplicates="drop")
        rate = tmp.groupby("Income_Group", observed=True)["Eligibility"].mean()
        plt.figure(figsize=(8,4)); plt.plot(range(len(rate)), rate.values, marker="o")
        plt.xticks(range(len(rate)), [str(x) for x in rate.index], rotation=45, ha="right")
        plt.title("Approval rate across applicant income groups"); plt.ylabel("Approval rate"); plt.xlabel("Income group"); plt.tight_layout(); plt.show()

## Practical 5 — Probability distributions and sampling methods

In [ ]:
# Examine a numeric feature's empirical distribution.
num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ["Eligibility"]]
if num_cols:
    dist_col = num_cols[0]
    vals = pd.to_numeric(df[dist_col], errors="coerce").dropna()
    plt.figure(figsize=(7,4)); plt.hist(vals, bins=30, density=True, alpha=.7)
    plt.title(f"Empirical distribution of {dist_col}"); plt.xlabel(dist_col); plt.ylabel("Density"); plt.show()
    print(f"{dist_col}: mean={vals.mean():.3f}, std={vals.std():.3f}, skewness={vals.skew():.3f}")

# Demonstrate simple random sampling and stratified sampling.
sample_n = min(10, len(df))
print("Simple random sample:")
display(df.sample(sample_n, random_state=RANDOM_STATE))
print("Stratified sample (proportional by eligibility class):")
stratified = df.groupby("Eligibility", group_keys=False).apply(
    lambda g: g.sample(max(1, int(round(sample_n*len(g)/len(df)))), random_state=RANDOM_STATE)
)
display(stratified.head(sample_n))

## Practical 6 — Central Limit Theorem (CLT) and point estimation

In [ ]:
if income_col:
    values = pd.to_numeric(df[income_col], errors="coerce").dropna().to_numpy()
    if len(values) >= 30:
        population_mean = values.mean()
        sample_means = []
        rng = np.random.default_rng(RANDOM_STATE)
        for _ in range(1000):
            sample = rng.choice(values, size=min(30, len(values)), replace=True)
            sample_means.append(sample.mean())
        plt.figure(figsize=(7,4)); plt.hist(sample_means, bins=30)
        plt.title("Sampling distribution of sample means (CLT demonstration)")
        plt.xlabel("Sample mean"); plt.ylabel("Frequency"); plt.show()
        print(f"Point estimate of mean {income_col}: {population_mean:.3f}")
        se = values.std(ddof=1) / np.sqrt(len(values))
        print(f"Approx. 95% confidence interval for mean: ({population_mean-1.96*se:.3f}, {population_mean+1.96*se:.3f})")
    else:
        print("Need at least 30 non-missing observations for this demonstration.")

## Practical 7 — Covariance, correlation and hypothesis testing

In [ ]:
corr_candidates = [c for c in df.select_dtypes(include=np.number).columns if c not in ["Eligibility"]]
if len(corr_candidates) >= 2:
    print("Covariance matrix:")
    display(df[corr_candidates].cov().round(3))
    print("Pearson correlation matrix:")
    display(df[corr_candidates].corr(method="pearson").round(3))
    print("Spearman correlation matrix:")
    display(df[corr_candidates].corr(method="spearman").round(3))

    a_col = income_col if income_col in corr_candidates else corr_candidates[0]
    b_col = loan_col if loan_col in corr_candidates and loan_col != a_col else corr_candidates[1]
    pair = df[[a_col, b_col]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(pair) >= 3:
        pearson_r, pearson_p = stats.pearsonr(pair[a_col], pair[b_col])
        spearman_r, spearman_p = stats.spearmanr(pair[a_col], pair[b_col])
        print(f"Pearson test ({a_col} vs {b_col}): r={pearson_r:.3f}, p-value={pearson_p:.5f}")
        print(f"Spearman test ({a_col} vs {b_col}): rho={spearman_r:.3f}, p-value={spearman_p:.5f}")
        print("At significance level 0.05, p < 0.05 indicates evidence of a non-zero association.")
else:
    print("Need at least two numeric columns for correlation tests.")

## Practical 8 — Train and compare machine-learning models

In [ ]:
# Remove target, original target column, label, and likely identifier columns from features.
drop_cols = {"Eligibility", "Eligibility_Label", TARGET}
drop_cols.update(c for c in df.columns if c.lower() in {"loan_id", "id", "customer_id", "applicant_id"})
X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
y = df["Eligibility"].astype(int)

# Remove columns that are completely empty.
X = X.dropna(axis=1, how="all")
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

transformers = []
if numeric_features:
    transformers.append(("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features))
if categorical_features:
    transformers.append(("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features))

if y.nunique() < 2:
    raise ValueError("Target must contain both approved and not-approved classes.")
if len(y) < 20:
    raise ValueError("Please use at least 20 rows to train and evaluate the models.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7)
}

trained_models = {}
results = []
for name, classifier in models.items():
    pipe = Pipeline([
        ("preprocess", ColumnTransformer(transformers, remainder="drop")),
        ("classifier", classifier)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    try:
        prob = pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob)
    except Exception:
        prob, auc = None, np.nan
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": auc
    })
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(results_df.round(3))
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
print("Selected model by F1-score:", best_model_name)

In [ ]:
# Confusion matrix and classification report for the selected model.
best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not eligible", "Eligible"])
disp.plot(values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()
print(classification_report(y_test, best_pred, target_names=["Not eligible", "Eligible"], zero_division=0))

if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    RocCurveDisplay.from_estimator(best_model, X_test, y_test)
    plt.title(f"ROC Curve — {best_model_name}")
    plt.show()

## Interactive applicant prediction form

In [ ]:
display(Markdown("Enter an applicant's details below. Dropdowns use observed categories from the dataset; numeric fields start at the dataset median."))

input_widgets = {}
form_items = []
for col in X.columns:
    if col in numeric_features:
        vals = pd.to_numeric(X[col], errors="coerce").dropna()
        default = float(vals.median()) if len(vals) else 0.0
        low = float(vals.min()) if len(vals) else default - 1
        high = float(vals.max()) if len(vals) else default + 1
        if low == high:
            low, high = low - 1, high + 1
        w = widgets.FloatText(value=round(default, 2), description=col[:18], style={"description_width":"initial"}, layout=widgets.Layout(width="95%"))
        # Numeric text inputs avoid artificially clipping new applicant values to observed extrema.
    else:
        vals = X[col].dropna().astype(str).unique().tolist()
        vals = sorted(vals)
        w = widgets.Dropdown(options=vals if vals else [""], value=vals[0] if vals else "", description=col[:18],
                             style={"description_width":"initial"}, layout=widgets.Layout(width="95%"))
    input_widgets[col] = w
    form_items.append(w)

predict_button = widgets.Button(description="Predict loan eligibility", button_style="primary", icon="check")
prediction_output = widgets.Output()
def run_prediction(_):
    with prediction_output:
        prediction_output.clear_output()
        row = pd.DataFrame([{c: w.value for c, w in input_widgets.items()}], columns=X.columns)
        pred = int(best_model.predict(row)[0])
        label = "LIKELY ELIGIBLE" if pred == 1 else "NOT LIKELY ELIGIBLE"
        print("Model:", best_model_name)
        print("Prediction:", label)
        if hasattr(best_model, "predict_proba"):
            prob = float(best_model.predict_proba(row)[0, 1])
            print(f"Estimated eligibility probability: {prob:.1%}")
        print("\nThis is an educational estimate based on the dataset—not a guarantee of loan approval.")
predict_button.on_click(run_prediction)

display(widgets.VBox(form_items))
display(predict_button, prediction_output)

## Optional — Download cleaned data and model metrics

In [ ]:
df.to_csv("cleaned_loan_eligibility_data.csv", index=False)
results_df.to_csv("loan_eligibility_model_comparison.csv", index=False)
files.download("cleaned_loan_eligibility_data.csv")
files.download("loan_eligibility_model_comparison.csv")

## Conclusion
This notebook demonstrates statistical analysis and machine-learning classification for loan eligibility. Results depend on the dataset and its labels. Check class balance, data quality, false approvals/rejections, and fairness before interpreting performance. Synthetic data is only for demonstrating the workflow.

## Final Practical — Launch the Streamlit Dashboard in Google Colab

This section runs the project as a **Streamlit web app**, not Gradio. Run the install cell, write the dashboard file, and then launch it. Colab needs a tunnel URL to open the Streamlit port in your browser; this notebook uses ngrok.

In [ ]:
%pip -q install streamlit pyngrok plotly scikit-learn pandas numpy

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

st.set_page_config(page_title="Bank Loan Eligibility Prediction", page_icon="🏦", layout="wide")
st.title("🏦 Bank Loan Eligibility Prediction")
st.caption("SI Lab project • Statistical analysis, model comparison, and applicant-level prediction")
st.info("Academic demonstration only. Predictions are not actual bank decisions.")

@st.cache_data
def make_demo_data(n=2200, seed=42):
    rng = np.random.default_rng(seed)
    gender = rng.choice(["Male", "Female"], n, p=[.68, .32])
    married = rng.choice(["Yes", "No"], n, p=[.62, .38])
    dependents = rng.choice(["0", "1", "2", "3+"], n, p=[.55, .18, .17, .10])
    education = rng.choice(["Graduate", "Not Graduate"], n, p=[.78, .22])
    self_employed = rng.choice(["No", "Yes"], n, p=[.86, .14])
    income = np.clip(rng.lognormal(np.log(5200), .55, n), 800, 30000).round()
    co_income = np.where(rng.random(n) < .35, 0, np.clip(rng.lognormal(np.log(1800), .7, n), 100, 15000)).round()
    loan_amount = np.clip(rng.lognormal(np.log(250), .75, n), 20, 2500).round()
    term = rng.choice([120, 180, 240, 300, 360, 480], n, p=[.03, .06, .08, .12, .62, .09])
    credit = rng.choice([0, 1], n, p=[.18, .82])
    area = rng.choice(["Rural", "Semiurban", "Urban"], n, p=[.28, .38, .34])
    total_income = income + co_income
    logit = -1.15 + 2.1*credit + .00008*(total_income-6000) - .004*(loan_amount-140) + .28*(education=="Graduate") + .30*(area=="Semiurban") - .25*(self_employed=="Yes")
    p = 1/(1+np.exp(-logit))
    status = np.where(rng.binomial(1, p) == 1, "Y", "N")
    data = pd.DataFrame({
        "Gender": gender, "Married": married, "Dependents": dependents,
        "Education": education, "Self_Employed": self_employed,
        "ApplicantIncome": income, "CoapplicantIncome": co_income,
        "LoanAmount": loan_amount, "Loan_Amount_Term": term,
        "Credit_History": credit, "Property_Area": area,
        "Loan_Status": status
    })
    for col in ["Gender", "Married", "LoanAmount", "Credit_History"]:
        idx = rng.choice(data.index, size=max(1, int(.02*n)), replace=False)
        data.loc[idx, col] = np.nan
    return data

def map_status(value):
    if pd.isna(value): return np.nan
    value = str(value).strip().lower()
    if value in {"y", "yes", "approved", "approve", "1", "true", "eligible"}: return 1
    if value in {"n", "no", "rejected", "reject", "0", "false", "not eligible", "ineligible"}: return 0
    try:
        number = float(value)
        if number in (0, 1): return int(number)
    except Exception:
        pass
    return np.nan

def prepare_data(raw):
    data = raw.copy()
    data.columns = [str(c).strip().replace(" ", "_") for c in data.columns]
    target_options = ["Loan_Status", "loan_status", "loan_approval", "LoanApproval", "Approved", "approved", "target", "Eligibility"]
    target = next((c for c in target_options if c in data.columns), None)
    if target is None:
        return None, None, None, "Could not find a target column. Use a target such as Loan_Status (Y/N)."
    data["Eligibility"] = data[target].apply(map_status)
    data = data.dropna(subset=["Eligibility"]).copy()
    if data["Eligibility"].nunique() < 2:
        return None, None, None, "Target must contain both eligible/approved and not eligible/rejected examples."
    data["Eligibility"] = data["Eligibility"].astype(int)
    data["Eligibility_Label"] = data["Eligibility"].map({1: "Eligible / Approved", 0: "Not eligible / Rejected"})
    drop_cols = {target, "Eligibility", "Eligibility_Label"}
    drop_cols.update(c for c in data.columns if c.lower() in {"id", "loan_id", "customer_id", "applicant_id"})
    X = data.drop(columns=[c for c in drop_cols if c in data.columns], errors="ignore")
    X = X.dropna(axis=1, how="all")
    y = data["Eligibility"]
    if len(data) < 20:
        return None, None, None, "Please upload at least 20 labelled rows."
    if X.empty:
        return None, None, None, "No usable feature columns found."
    return data, X, y, None

with st.sidebar:
    st.header("Data")
    uploaded = st.file_uploader("Upload loan approval CSV (optional)", type=["csv"])
    if uploaded is not None:
        try:
            raw = pd.read_csv(uploaded)
            source = "Uploaded CSV"
        except Exception as exc:
            st.error(f"Could not read CSV: {exc}")
            raw, source = make_demo_data(), "Synthetic demo data"
    else:
        raw, source = make_demo_data(), "Synthetic demo data"
    st.caption(f"Data source: {source}")

data, X, y, err = prepare_data(raw)
if err:
    st.error(err)
    st.stop()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
transformers = []
if numeric_features:
    transformers.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features))
if categorical_features:
    transformers.append(("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features))
preprocessor = ColumnTransformer(transformers, remainder="drop")

try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
except ValueError as exc:
    st.error(f"Unable to create train/test split: {exc}")
    st.stop()

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7)
}
trained, rows = {}, []
for name, classifier in classifiers.items():
    model = Pipeline([("preprocess", preprocessor), ("classifier", classifier)])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    try:
        probabilities = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probabilities)
    except Exception:
        auc = np.nan
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": auc
    })
    trained[name] = model
metrics = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
best_name = metrics.iloc[0]["Model"]
best_model = trained[best_name]

tab_overview, tab_stats, tab_models, tab_predict, tab_data = st.tabs([
    "📊 Overview", "📐 Statistics", "🤖 Model Comparison", "🧮 Predict Eligibility", "🗂️ Data"
])

with tab_overview:
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Applicant records", f"{len(data):,}")
    c2.metric("Eligible / approved", f"{int((y == 1).sum()):,}")
    c3.metric("Not eligible / rejected", f"{int((y == 0).sum()):,}")
    c4.metric("Approval rate", f"{y.mean():.1%}")
    left, right = st.columns(2)
    counts = data["Eligibility_Label"].value_counts().rename_axis("Status").reset_index(name="Applicants")
    left.plotly_chart(px.pie(counts, names="Status", values="Applicants", hole=.45, title="Loan approval distribution"), use_container_width=True)
    right.plotly_chart(px.histogram(data, x="LoanAmount", color="Eligibility_Label" if "Eligibility_Label" in data else None, title="Loan amount distribution"), use_container_width=True)
    if {"ApplicantIncome", "LoanAmount"}.issubset(data.columns):
        st.plotly_chart(px.scatter(data, x="ApplicantIncome", y="LoanAmount", color="Eligibility_Label", title="Applicant income vs loan amount"), use_container_width=True)

with tab_stats:
    st.subheader("Descriptive statistics")
    numeric = data.select_dtypes(include=np.number)
    if not numeric.empty:
        summary = numeric.describe().T
        summary["median"] = numeric.median(numeric_only=True)
        summary["mode"] = numeric.mode().iloc[0]
        summary["variance"] = numeric.var(numeric_only=True)
        summary["Q1"] = numeric.quantile(.25)
        summary["Q3"] = numeric.quantile(.75)
        summary["IQR"] = summary["Q3"] - summary["Q1"]
        st.dataframe(summary.round(3), use_container_width=True)
        numeric_cols = [c for c in numeric.columns if c != "Eligibility"]
        if numeric_cols:
            col = st.selectbox("Choose numeric feature", numeric_cols)
            c1, c2 = st.columns(2)
            c1.plotly_chart(px.histogram(data, x=col, marginal="box", title=f"Distribution of {col}"), use_container_width=True)
            c2.plotly_chart(px.box(data, y=col, title=f"Box plot of {col}"), use_container_width=True)

with tab_models:
    st.subheader("Held-out test-set model comparison")
    st.dataframe(metrics.round(3), use_container_width=True)
    selected = st.selectbox("Confusion matrix for model", list(trained.keys()), index=list(trained.keys()).index(best_name))
    pred = trained[selected].predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    st.plotly_chart(px.imshow(cm, text_auto=True, x=["Predicted not eligible", "Predicted eligible"], y=["Actual not eligible", "Actual eligible"], title=f"Confusion matrix — {selected}"), use_container_width=True)
    st.caption(f"Best model selected by F1-score: {best_name}")

with tab_predict:
    st.subheader("Check whether an applicant is likely to be eligible")
    st.write("Enter the applicant's details. Loan amount accepts large values; the form is not limited to the training data's observed maximum.")
    values = {}
    with st.form("eligibility_form"):
        cols = st.columns(3)
        for i, feature in enumerate(X.columns):
            container = cols[i % 3]
            if feature in numeric_features:
                vals = pd.to_numeric(X[feature], errors="coerce").dropna()
                default = float(vals.median()) if len(vals) else 0.0
                if feature.lower() in {"loanamount", "loan_amount", "loan_amount_thousands"}:
                    values[feature] = container.number_input(feature.replace("_", " ").title(), min_value=0.0, max_value=100000000.0, value=max(0.0, default), step=10.0, help="Enter a loan amount up to 100,000,000 in your dataset's units.")
                else:
                    values[feature] = container.number_input(feature.replace("_", " ").title(), value=default, step=1.0)
            else:
                options = sorted(X[feature].dropna().astype(str).unique().tolist())
                if not options:
                    options = [""]
                default = str(X[feature].dropna().astype(str).mode().iloc[0]) if X[feature].notna().any() else options[0]
                if default not in options:
                    default = options[0]
                values[feature] = container.selectbox(feature.replace("_", " ").title(), options, index=options.index(default))
        submitted = st.form_submit_button("Predict loan eligibility", type="primary", use_container_width=True)
    if submitted:
        applicant = pd.DataFrame([values], columns=X.columns)
        prediction = int(best_model.predict(applicant)[0])
        try:
            probability = float(best_model.predict_proba(applicant)[0, 1])
        except Exception:
            probability = None
        if prediction == 1:
            st.success("✅ RESULT: LIKELY ELIGIBLE FOR A LOAN")
        else:
            st.error("❌ RESULT: NOT LIKELY ELIGIBLE FOR A LOAN")
        if probability is not None:
            st.metric("Estimated eligibility probability", f"{probability:.1%}")
        st.write(f"Model used: **{best_name}**")
        st.caption("This is a model estimate based on historical labels, not a guarantee of actual bank approval.")

with tab_data:
    st.subheader("Cleaned dataset")
    st.dataframe(data.head(500), use_container_width=True)
    st.download_button("Download cleaned CSV", data.to_csv(index=False).encode("utf-8"), "cleaned_loan_eligibility.csv", "text/csv")
    st.download_button("Download model comparison CSV", metrics.to_csv(index=False).encode("utf-8"), "loan_model_metrics.csv", "text/csv")


### Start Streamlit

When prompted, paste your own ngrok authtoken. The input is hidden. Do not share your token with anyone. If you already started an old Streamlit process in this runtime, restart the runtime before running this cell.

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import subprocess, time, socket, os

# Enter the token from your ngrok dashboard. The token will not be displayed.
ngrok.set_auth_token(getpass("Enter your ngrok authtoken: "))

# Close any ngrok tunnels created in this runtime, then launch Streamlit.
try:
    ngrok.kill()
except Exception:
    pass

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.address", "0.0.0.0", "--server.headless", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit to start listening.
ready = False
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", 8501), timeout=1):
            ready = True
            break
    except OSError:
        time.sleep(1)

if not ready:
    raise RuntimeError("Streamlit did not start on port 8501. Restart the runtime and run this cell again.")
public_url = ngrok.connect(8501, "http")
print("Your Streamlit dashboard is ready:")
print(public_url)